# Phase 1 — Denoising Autoencoder Pretraining
**Continuous Sign Language Translation** · Phase 1 of 2

Trains a `MultiStreamSemanticEncoder` + `StructuredDiffusionDecoder` using a proper DDPM noise schedule with epsilon prediction.

The encoder learns a compact latent `Z [B, T/4, 512]` that will be used in Phase 2 for translation.

**Runtime:** GPU (T4 or better)


In [1]:
# ── Clone repository and install dependencies ──────────────────────────
!git clone https://github.com/bencejdanko/cslt-flan-t5-small-autoencoder.git cslt
%cd cslt
!git pull origin main
!pip install -q -r requirements.txt

fatal: destination path 'cslt' already exists and is not an empty directory.
/content/cslt
From https://github.com/bencejdanko/cslt-flan-t5-small-autoencoder
 * branch            main       -> FETCH_HEAD
Already up to date.


In [2]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [3]:
# ── Hugging Face authentication ───────────────────────────────────────
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets ✓")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("Paste your HF token: ")

import os
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

HF_TOKEN loaded from Colab Secrets ✓


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## ⚙️ Configuration
Adjust these before running. All parameters are passed to the `phase1_pretrain` function via `Phase1Config`.


In [6]:
from config import Phase1Config

cfg = Phase1Config(
    max_samples=100,          # Set to None for full dataset
    batch_size=32,
    epochs=3,
    lr=1e-4,
    ckpt_dir="/content/phase1_ckpt",
    upload_hf=False,           # Set to True to auto-upload checkpoints
    hf_repo="bdanko/cslt-flan-t5-small-autoencoder",
    seed=42,
)
print(f"Config: {cfg}")

Config: Phase1Config(dataset_repo='bdanko/how2sign-landmarks-front-raw-parquet', split='train', max_samples=100, val_max_samples=20, batch_size=32, d_model=384, latent_dim=512, encoder_layers=3, encoder_heads=8, encoder_dropout=0.1, use_part_embeddings=True, ddpm=DDPMConfig(num_timesteps=1000, beta_start=0.0001, beta_end=0.02, schedule_type='linear'), masking=MaskingConfig(feature_corruption=True, time_span_masking=True, whole_part_masking=True, velocity_reconstruction=True, latent_smoothness=True, contrastive_consistency=False, feature_corruption_prob=0.15, time_span_ratio=0.2, contrastive_weight=0.05), w_masked_pos=1.0, w_masked_vel=1.0, w_full_recon=0.1, w_latent_smooth=0.01, w_contrastive=0.05, epochs=3, lr=0.0001, weight_decay=1e-05, grad_clip=1.0, scheduler='cosine', warmup_steps=100, mixed_precision=False, gradient_accumulation_steps=1, ckpt_dir='/content/phase1_ckpt', hf_repo='bdanko/cslt-flan-t5-small-autoencoder', upload_hf=True, seed=42, log_backend='csv', wandb_project='csl

## Training
Runs the Phase 1 pretraining loop with DDPM epsilon prediction, configurable masking, train/val splits, and best checkpoint saving.

In [5]:
from phase1_pretrain import train_phase1

train_phase1(cfg)

Train E1: 0it [00:00, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Val E1: 0it [00:00, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


...

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Val E3: 0it [00:00, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

## Upload to Hugging Face (optional)

In [7]:
from huggingface_hub import HfApi, create_repo, upload_folder

if cfg.upload_hf:
    create_repo(cfg.hf_repo, token=HF_TOKEN, exist_ok=True)
    upload_folder(
        folder_path=cfg.ckpt_dir,
        repo_id=cfg.hf_repo,
        token=HF_TOKEN,
        commit_message=f"Phase 1 checkpoint — {cfg.epochs} epochs, {cfg.max_samples} clips",
    )
    print(f"Uploaded to https://huggingface.co/{cfg.hf_repo} ✓")
else:
    print("Skipping upload. Set cfg.upload_hf = True to upload.")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e1_ckpt/best/optimizer.pt:  35%|###4      | 32.0MB / 92.0MB            

  ..._ckpt/latest/optimizer.pt:   0%|          | 8.88kB / 92.0MB            

  ...ase1_ckpt/best/encoder.pt:  45%|####4     | 15.9MB / 35.5MB            

  ...phase1_ckpt/best/model.pt:  45%|####4     | 23.9MB / 53.7MB            

  ...ase1_ckpt/latest/model.pt:  14%|#4        | 7.59MB / 53.7MB            

  ...e1_ckpt/latest/encoder.pt:  21%|##1       | 7.59MB / 35.5MB            

Uploaded to https://huggingface.co/bdanko/cslt-flan-t5-small-autoencoder ✓
